In [47]:
from QASMBench.interface import qiskit

# path to the root directory of QASMBench
path = "QASMBench"

    # selected category for QASMBench
category = "small" 

    # select only the circuits with the number of qubits in the list
num_qubits_list = list(range(0, 100))

    # whether to remove the final measurement in the circuit
remove_final_measurements = True

    # whether use qiskit.transpile() to transpile the circuits (note: must provide qiskit backend)
do_transpile = False

    # arguments for qiskit.transpile(). backend should be provide at least
transpile_args = {}
    
bm = qiskit.QASMBenchmark(path, category, num_qubits_list=num_qubits_list, remove_final_measurements=remove_final_measurements, do_transpile=do_transpile, **transpile_args)
#print(bm)    

from quantum_optimiser import integration
from qiskit import transpile
import pyzx 
circuit = bm.get("ising_n10")
print(circuit)
diagram = integration.qiskit_to_pyzx(circuit)
pyzx.draw(diagram, labels="true")

       ┌───┐ ┌──────────┐                                        ┌───┐    »
reg_0: ┤ H ├─┤ Rz(-0.3) ├───────────────■─────────────────■──────┤ H ├────»
       ├───┤ ├─────────┬┘ ┌─────────┐ ┌─┴─┐ ┌──────────┐┌─┴─┐ ┌──┴───┴───┐»
reg_1: ┤ H ├─┤ Rz(0.3) ├──┤ Rz(0.3) ├─┤ X ├─┤ Rz(-0.3) ├┤ X ├─┤ Rz(0.26) ├»
       ├───┤┌┴─────────┴┐ └─────────┘ └───┘ └──────────┘└───┘┌┴──────────┤»
reg_2: ┤ H ├┤ Rz(-0.36) ├───────────────■─────────────────■──┤ Rz(-0.26) ├»
       ├───┤└┬──────────┤ ┌──────────┐┌─┴─┐┌───────────┐┌─┴─┐├───────────┤»
reg_3: ┤ H ├─┤ Rz(0.36) ├─┤ Rz(0.36) ├┤ X ├┤ Rz(-0.36) ├┤ X ├┤ Rz(-0.26) ├»
       ├───┤┌┴──────────┤ └──────────┘└───┘└───────────┘└───┘└┬──────────┤»
reg_4: ┤ H ├┤ Rz(-0.12) ├───────────────■─────────────────■───┤ Rz(0.26) ├»
       ├───┤└┬──────────┤ ┌──────────┐┌─┴─┐┌───────────┐┌─┴─┐ ├──────────┤»
reg_5: ┤ H ├─┤ Rz(0.12) ├─┤ Rz(0.12) ├┤ X ├┤ Rz(-0.12) ├┤ X ├─┤ Rz(0.38) ├»
       ├───┤ ├──────────┤ └──────────┘└───┘└───────────┘└───┘┌┴──────────┤»
reg_6: ┤ H ├

In [48]:
from quantum_optimiser.multimetric import loss, simulated_annealing
from quantum_optimiser import integration
diagram = integration.qiskit_to_pyzx(circuit)
pyzx.draw(diagram, labels="true")
circuit = transpile(circuit,basis_gates=["cx", "h", "t", "tdg", "s", "sdg", "x", "y", "z","rz"],optimization_level=0)

print("Initial stats:", loss._compute_stats(circuit))

circ2 = integration.pyzx_to_qiskit(diagram)
print("final stats:", loss._compute_stats(circ2))


Initial stats: {'q': 10, 'g2': 90, 'g': 480, 't': 0, 'd': 70, 'swap': 0}
final stats: {'q': 10, 'g2': 90, 'g': 460, 't': 0, 'd': 68, 'swap': 0}


In [49]:

from quantum_optimiser.multimetric import loss, simulated_annealing
from quantum_optimiser.hardware_aware import hardware_map
from quantum_optimiser.hardware_aware import hardwares
hardware=hardwares.sycamore

import pyzx
print("Initial stats:", loss._compute_stats(circuit))

optimised, diagram, _, history = simulated_annealing.simulated_annealing_zx(
    circuit=circuit,
    cost_function=loss.log_weighted_loss,
    get_neighbor=simulated_annealing.get_neighbor_weighted_rules,
    initial_temp=100.0,
    cooling_rate=0.99,
    max_iterations=2000,
    max_no_improvement=20,
    hardware=hardware
)

print("Final stats:", loss._compute_stats(optimised))



Initial stats: {'q': 10, 'g2': 90, 'g': 480, 't': 0, 'd': 70, 'swap': 0}
our baseline is: {'q': 10, 'g2': 90, 'g': 480, 't': 0, 'd': 70, 'swap': 0}
had to go here again
had to go here again
had to go here again
we have an improvement
had to go here again
had to go here again
had to go here again
we have an improvement
we have an improvement
had to go here again
had to go here again
had to go here again
had to go here again
had to go here again
had to go here again
we have an improvement
had to go here again
had to go here again
had to go here again
we have an improvement
we have an improvement
we have an improvement
we have an improvement
had to go here again
we have an improvement
had to go here again
had to go here again
we have an improvement
had to go here again
we have an improvement
had to go here again
had to go here again
had to go here again
had to go here again
had to go here again
we have an improvement
had to go here again
had to go here again
had to go here again
had to go

In [50]:
from qiskit import transpile
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import CouplingMap
sycamore_edges = [
    # Row 0
    (0, 1),
    # Row 1
    (1, 2), (2, 3),
    # Row 2
    (3, 4), (4, 5), (5, 6), (6, 7),
    # Row 3
    (7, 8), (8, 9), (9, 10), (10, 11), (11, 12),
    # Row 4
    (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18),
    # Row 5
    (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25),
    # Row 6
    (25, 26), (26, 27), (27, 28), (28, 29), (29, 30), (30, 31),
    # Row 7
    (31, 32), (32, 33), (33, 34), (34, 35), (35, 36),
    # Row 8
    (36, 37), (37, 38), (38, 39), (39, 40),
    # Row 9
    (40, 41), (41, 42), (42, 43),
    # Row 10
    (43, 44), (44, 45),
    # Row 11
    (45, 46),
    # Vertical connections between rows
    (0, 3), (1, 4), (2, 5), (3, 6), (4, 7), (5, 8), (6, 9),
    (7, 12), (8, 13), (9, 14), (10, 15), (11, 16), (12, 17),
    (13, 18), (14, 19), (15, 20), (16, 21), (17, 22), (18, 23),
    (19, 25), (20, 26), (21, 27), (22, 28), (23, 29), (24, 30),
    (25, 31), (26, 32), (27, 33), (28, 34), (29, 35), (30, 36),
    (31, 37), (32, 38), (33, 39), (34, 40),
    (36, 41), (37, 42), (38, 43),
    (40, 44), (41, 45),
    (43, 46),
    (46, 47), (47, 48), (48, 49), (49, 50), (50, 51), (51, 52),
    (44, 47), (45, 48), (46, 49), (47, 50), (48, 51), (49, 52),
]
coupling_map = CouplingMap(sycamore_edges)  
mapped_only = transpile(
    circuit,
    basis_gates=["cx", "h", "t", "tdg", "s", "sdg", "x", "y", "z", "rz","swap"],
    coupling_map=coupling_map,
    layout_method="sabre",
    routing_method="sabre",
    optimization_level=3,
)

print(loss._compute_stats(mapped_only))

{'q': 10, 'g2': 90, 'g': 467, 't': 0, 'd': 103, 'swap': 0}


In [59]:
# test if your rewrite rules can actually reduce g2
import pyzx
from quantum_optimiser import integration
from quantum_optimiser.multimetric import loss

diagram = integration.qiskit_to_pyzx(circuit)
print("before:", loss._compute_stats(integration.pyzx_to_qiskit(diagram)))

# try pyzx's own simplification directly
test = diagram.copy()
pyzx.simplify.full_reduce(test)
result = integration.pyzx_to_qiskit(test)
print("after full_reduce:", loss._compute_stats(result))

before: {'q': 26, 'g2': 50, 'g': 228, 't': 0, 'd': 13, 'swap': 0}
after full_reduce: {'q': 26, 'g2': 650, 'g': 2077, 't': 0, 'd': 1101, 'swap': 0}


In [1]:
import random
from qiskit import QuantumCircuit
from quantum_optimiser.multimetric import rewrite
from quantum_optimiser import integration
import pyzx 
    

one_qubit_gates = ["h", "s", "sdg", "t", "tdg", "x", "z"]
two_qubit_gates = ["cx","cz",]

def random_circuit_generator(max_qubits, max_depth):
    qubits = random.randint(2, max_qubits)
    depth = random.randint(2, max_depth)
    qc = QuantumCircuit(qubits)

          
    for i in range(depth):
        if random.random() < 0.6:
            q = random.randrange(qubits)
            gate = random.choice(one_qubit_gates)
            getattr(qc, gate)(q)
        else:
            q1, q2 = random.sample(range(qubits), 2)
            gate = random.choice(two_qubit_gates)
            getattr(qc, gate)(q1, q2)

    return qc


qc = random_circuit_generator(20,10)

        

In [2]:
from quantum_optimiser.hardware_aware import hardware_map
from quantum_optimiser.hardware_aware import hardwares
hardware=hardwares.sherbrooke
from quantum_optimiser.multimetric import loss
from quantum_optimiser.multimetric import simulated_annealing
circuit = integration.pyzx_to_qiskit(integration.qiskit_to_pyzx(qc))

print("pre mapping:", loss._compute_stats(qc))

circuit = hardware.make_compatible(circuit)
print("Initial:", loss._compute_stats(circuit))


# SA
optimised, _, _, history = simulated_annealing.simulated_annealing_zx(
    circuit=qc,
    cost_function=loss.log_weighted_loss,
    get_neighbor=simulated_annealing.get_neighbor_weighted_rules,
    initial_temp=100.0,
    cooling_rate=0.95,
    max_iterations=500,
    max_no_improvement=50,
    hardware=hardware
)

print("Final:", loss._compute_stats(optimised))



pre mapping: {'q': 5, 'g2': 2, 'g': 4, 't': 0, 'd': 2}
Initial: {'q': 14, 'g2': 24, 'g': 28, 't': 0, 'd': 24}
Final: {'q': 14, 'g2': 24, 'g': 28, 't': 0, 'd': 24}


In [6]:
from quantum_optimiser import optimiser
from quantum_optimiser.optimiser import evaluate



from quantum_optimiser.hardware_aware import hardwares
hardware = hardwares.sherbrooke 

evaluate(qubits=100, n_circuits=50, depth=50, hardware=hardware)


Benchmark over 50 circuits | qubits≤100 depth≤50

naive
  Avg cost decrease: 10.3000
  Avg qubits decrease: 0.3000
  Avg two_qubit decrease: 2.8400
  Avg gates decrease: 1.4600
  Avg depth decrease: 2.3000
  Avg t decrease: 0.3000

informed
  Avg cost decrease: 5.7580
  Avg qubits decrease: 0.0200
  Avg two_qubit decrease: 3.3600
  Avg gates decrease: -1.9200
  Avg depth decrease: 2.9200
  Avg t decrease: 0.1600

log_weighted
  Avg cost decrease: 49.3642
  Avg qubits decrease: -2.5600
  Avg two_qubit decrease: 5.8800
  Avg gates decrease: -57.5800
  Avg depth decrease: -2.5000
  Avg t decrease: 0.4400

quadratic
  Avg cost decrease: 108.3701
  Avg qubits decrease: -0.3600
  Avg two_qubit decrease: 4.7200
  Avg gates decrease: -4.9800
  Avg depth decrease: 2.6800
  Avg t decrease: 0.1800

